# Starting from zero: probability and stochastic motion

**Audience:** no previous stochastic-process course is assumed. You need basic derivatives, integrals and Python arrays. Read the explanatory text, predict each result, run the cell, and then answer the questions. See [`notes/00_foundations.md`](../notes/00_foundations.md) for the full introduction and [`notes/derivations.md`](../notes/derivations.md) for derivations.

We use dimensionless units. A seeded pseudo-random generator makes this notebook reproducible; its output still represents random samples, not exact ensemble averages. Run cells in order.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from scipy.stats import norm

rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "font.size": 11})

## 1. Distribution, mean, and standard error

A probability density $p(x)$ is normalized: integrating it between two points gives a probability. A density has units of inverse $x$. Here $Z$ is standard normal with $\mathbb E Z=0$ and $\mathrm{Var}(Z)=1$. The standard deviation measures variation among individual draws, whereas $s/\sqrt N$ measures uncertainty in the sample mean for independent draws.

**Predict:** which quantity becomes smaller when $N$ increases? Why is a histogram not perfectly smooth?

In [ ]:
n_samples = 10_000
samples = rng.normal(loc=0.0, scale=1.0, size=n_samples)
mean = samples.mean()
standard_deviation = samples.std(ddof=1)
standard_error = standard_deviation / np.sqrt(n_samples)
print(f"mean = {mean:.4f}; sample SD = {standard_deviation:.4f}")
print(f"standard error of the mean ≈ {standard_error:.4f}")
xx = np.linspace(-4, 4, 400)
plt.hist(samples, bins=45, density=True, alpha=0.5, label="samples")
plt.plot(xx, norm.pdf(xx), label="standard normal density")
plt.xlabel("value"); plt.ylabel("probability density"); plt.legend(); plt.show()

**Try it:** change `n_samples` to 100 and then 100,000. Repeat the cell: the random stream advances. The mean and its uncertainty change, but the population variance remains one. Correlated frames require an uncertainty method that accounts for correlations; the independent-sample formula is not generally valid for a trajectory.

## 2. Stochastic processes and Brownian paths

A stochastic process $X_t$ is a family of random variables indexed by time. One curve is a sample path; a histogram at fixed $t$ describes an ensemble. For standard Brownian motion, non-overlapping increments are independent and $\Delta W\sim\mathcal N(0,\Delta t)$. Thus

$$X_{n+1}=X_n+\sqrt{2D\Delta t}\,\xi_n,\quad\xi_n\sim\mathcal N(0,1),$$

and $\mathbb E[X_t]=x_0$, $\mathrm{Var}(X_t)=2Dt$. The $\sqrt{\Delta t}$ factor is essential. **Predict:** how do the mean and variance change when $D$ doubles?

In [ ]:
D = 0.8                 # length^2/time
initial_position = 0.0
step = 0.01             # time
n_steps = 300
n_paths = 4000
increments = np.sqrt(2 * D * step) * rng.normal(size=(n_paths, n_steps))
paths = np.empty((n_paths, n_steps + 1))
paths[:, 0] = initial_position
paths[:, 1:] = initial_position + np.cumsum(increments, axis=1)
times = np.arange(n_steps + 1) * step
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(times, paths[:12].T, alpha=0.7)
axes[0].set(xlabel="time", ylabel="position", title="12 realizations")
axes[1].plot(times, paths.var(axis=0), label="ensemble variance")
axes[1].plot(times, 2 * D * times, "--", label="2Dt")
axes[1].set(xlabel="time", ylabel="variance"); axes[1].legend()
plt.tight_layout(); plt.show()

**Check:** the $N$ values at each time come from independent paths. If you replace $\sqrt{\Delta t}$ by $\Delta t$, does the variance converge to the correct expression as the time step decreases? No: at fixed time it incorrectly tends to zero. Explore this numerically.

## 3. Drift, diffusion and an exactly solvable model

For $dX_t=a(X_t)dt+\sqrt{2D}dW_t$, the drift $a$ controls the mean local displacement and $D$ the local variance. In a harmonic well $F(x)=\frac12\kappa x^2$, the equilibrium Itô SDE is

$$dX_t=-D\beta\kappa X_t\,dt+\sqrt{2D}\,dW_t.$$

Let $\lambda=D\beta\kappa$. For a fixed initial position $x_0$, the exact moments are

$$\mathbb E[X_t]=x_0e^{-\lambda t},\quad \mathrm{Var}(X_t)=\frac{D}{\lambda}(1-e^{-2\lambda t}).$$

These follow by solving the linear SDE, or using Itô's formula for $f(x)=x$ and $x^2$. The next cell compares numerical Euler–Maruyama moments with these exact expressions. **Predict:** where does the mean go, and what is the long-time variance?

In [ ]:
D, beta, kappa = 1.0, 1.0, 2.0
rate = D * beta * kappa
step, n_steps, n_paths = 0.002, 1000, 4000
initial_position = 1.5
x = np.full(n_paths, initial_position)
mean = np.empty(n_steps + 1); variance = np.empty(n_steps + 1)
mean[0], variance[0] = x.mean(), x.var()
for i in range(1, n_steps + 1):
    gaussian_increment = rng.normal(size=n_paths)
    x += -rate * x * step + np.sqrt(2 * D * step) * gaussian_increment
    mean[i], variance[i] = x.mean(), x.var()
times = np.arange(n_steps + 1) * step
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(times, mean, label="simulation")
axes[0].plot(times, initial_position*np.exp(-rate*times), "--", label="exact")
axes[0].set(xlabel="time", ylabel="mean"); axes[0].legend()
axes[1].plot(times, variance, label="simulation")
axes[1].plot(times, D/rate*(1-np.exp(-2*rate*times)), "--", label="exact")
axes[1].set(xlabel="time", ylabel="variance"); axes[1].legend()
plt.tight_layout(); plt.show()

**Interpretation:** $D/\lambda=(\beta\kappa)^{-1}=k_BT/\kappa$ is the equilibrium variance. At fixed temperature and well stiffness, changing $D$ changes relaxation time $1/\lambda$ but not the equilibrium variance. Euler–Maruyama is approximate: halve `step` at fixed total time and increase `n_paths` to assess time-discretization and sampling error separately.

## 4. From paths to densities: forward and backward equations

For the Itô SDE $dX=a(X)dt+\sqrt{2D(X)}dW$, the backward generator acting on a smooth test function is $\mathcal Lf=af'+Df''$. Itô's formula gives $d\mathbb E f(X_t)/dt=\mathbb E[\mathcal L f(X_t)]$. Integrating by parts gives the forward equation

$$\partial_t p=-\partial_x(ap)+\partial_x^2(Dp)=-\partial_xJ,\qquad J=ap-\partial_x(Dp).$$

With constant $D$ and equilibrium drift $a=-D\beta F'$, $J=-D(\beta F'p+p')$; setting $J=0$ gives $p_\mathrm{eq}\propto e^{-\beta F}$. Boundary terms require care: reflecting means $J=0$, while regular absorbing boundaries remove probability.

**Work it out:** insert $f(x)=x^2$ into $\mathcal L f$ for the harmonic model, derive the ODE for $\mathbb E X_t^2$, and combine it with the mean to obtain the variance shown above. `notes/00_foundations.md` contains intermediate steps and assumptions.

## 5. First passage and censoring

The first-passage time $\tau_b$ is the first time a trajectory reaches the target $b$. Its survival $S(t)=P(\tau_b>t)$ is the fraction not yet arrived. For a nonnegative $\tau_b$, its mean is $\int_0^\infty S(t)dt$ when finite. Stopping a run at time $T$ yields an observed time $\min(\tau_b,T)$ and a hit flag; no-hit paths are *right-censored*. Their mean estimates the restricted mean through $T$, not the full MFPT.

For an exactly solvable example without censoring, generate exponential passage times with mean 4. This is an illustration of survival analysis; diffusion first-passage times are not in general exponential.

In [ ]:
event_times = rng.exponential(scale=4.0, size=5000)
horizon = 3.0
observed = np.minimum(event_times, horizon)
hit = event_times <= horizon
times = np.linspace(0, horizon, 200, endpoint=False)
survival = np.array([np.mean(observed > t) for t in times])
plt.plot(times, survival, label="empirical, before censoring time")
plt.plot(times, np.exp(-times/4), "--", label="exact survival")
plt.xlabel("time"); plt.ylabel("survival probability"); plt.legend(); plt.show()
print(f"censored fraction: {(~hit).mean():.3f}")
print(f"restricted mean: {observed.mean():.3f}; exact: {4*(1-np.exp(-horizon/4)):.3f}")
print(f"mean of hits only: {observed[hit].mean():.3f}; full true mean: 4")

**Question:** Why is the observed-events-only mean too small here? Does the empirical curve at a finite censoring horizon determine the unobserved tail? Answer: late events are underrepresented, and the tail requires additional data or assumptions.

Continue with `01_student.ipynb`: there you will specify a double-well FPT experiment, derive the backward equation, and compare direct simulation with quadrature and a discrete generator. The optional committor asks which of two states is reached first.